# Notebook 1 — Krylov methods for the Hessian

ARENA-style hands-on tutorial on matrix-free eigenvalue methods, with the
loss Hessian of a tiny neural network as the running example.

**Time:** ~145 min.  **Prerequisites:** PyTorch, basic autodiff.

**Sections:**
0. NLA you'll need (norms, condition number, machine epsilon, Rayleigh quotient)
1. The matvec is the unit of cost: HVP three ways
2. Power iteration
3. Deflation
4. Lanczos: the three-term recurrence
5. Loss of orthogonality, and how to fix it
6. Hessian top-k in practice

Solutions live in `solutions/_01_krylov.py`.  Inline tests live in `src/tests.py`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import matplotlib.pyplot as plt
import numpy as np

from src import tests
from src.plotting import apply_style, semilog_convergence, eigenvalue_compare
from src.tiny_models import toy_mlp, tiny_mlp, count_params

apply_style()
torch.manual_seed(0)
print('environment ready')

## 0. NLA you'll need

This section establishes the four concepts the rest of the notebook leans on.
We state, don't derive — proofs in any numerical-linear-algebra textbook.

### 0.1 Operator vs Frobenius norm

For a matrix $A \in \mathbb{R}^{m \times n}$:

- **Operator norm** $\|A\|_2 = \sigma_{\max}(A)$.  Controls the worst-case
  matvec amplification: $\|Av\|_2 \le \|A\|_2 \|v\|_2$.
- **Frobenius norm** $\|A\|_F = \sqrt{\sum_{ij} A_{ij}^2} = \sqrt{\sum_i \sigma_i^2}$.
  Controls the *variance* of stochastic trace estimators.

Always $\|A\|_2 \le \|A\|_F \le \sqrt{\text{rank}(A)} \cdot \|A\|_2$.

In [ ]:
# Exercise 0.1 (🔴⚪⚪⚪⚪, 3 min)
# Compute both norms of a 20x20 random matrix, verify the inequality.
torch.manual_seed(0)
A = torch.randn(20, 20)

op_norm = None  # YOUR CODE HERE: compute ||A||_2
fro_norm = None  # YOUR CODE HERE: compute ||A||_F

assert op_norm is not None and fro_norm is not None, 'fill in op_norm and fro_norm'
assert op_norm <= fro_norm <= (20 ** 0.5) * op_norm
print(f'||A||_2 = {op_norm:.3f}, ||A||_F = {fro_norm:.3f}')

### 0.2 Condition number

For a square invertible $A$: $\kappa(A) = \sigma_{\max}/\sigma_{\min}$.
For symmetric positive-definite $A$ this is $\lambda_{\max}/\lambda_{\min}$.

**Why it matters:** in `fp32`, solving $A x = b$ loses about $\log_{10}\kappa$
digits.  Iterative methods converge at a rate that depends on the eigenvalue
*gap*, which is bounded by $\kappa$.

Below: we make $\kappa$ progressively worse and watch the residual blow up.

In [ ]:
# Exercise 0.2 (🔴🔴⚪⚪⚪, 8 min)
# Build a symmetric PSD A with engineered eigenvalues {1, 1, ..., 1, 1/kappa},
# solve Ax = b in fp32, plot relative residual vs kappa on log-log axes.

def make_conditioned_spd(n: int, kappa: float, seed: int = 0):
    g = torch.Generator().manual_seed(seed)
    Q = torch.linalg.qr(torch.randn(n, n, generator=g))[0]
    eigs = torch.ones(n)
    eigs[-1] = 1.0 / kappa
    return Q @ torch.diag(eigs) @ Q.T

n = 30
kappas = torch.logspace(0, 8, 9)
residuals = []
for kappa in kappas:
    A = make_conditioned_spd(n, kappa.item()).float()
    x_true = torch.randn(n)
    b = A @ x_true
    x_hat = None  # YOUR CODE HERE: solve A x_hat = b in fp32
    assert x_hat is not None, 'solve A x_hat = b using torch.linalg.solve'
    residuals.append(((A @ x_hat - b).norm() / b.norm()).item())

plt.figure()
plt.loglog(kappas.numpy(), residuals, 'o-')
plt.xlabel(r'$\kappa(A)$'); plt.ylabel('relative residual')
plt.title('fp32 solve precision vs condition number')
plt.show()

### 0.3 Machine epsilon

`fp32` has $\epsilon_{\text{mach}} \approx 1.19 \times 10^{-7}$.
That's the smallest $x$ such that $1 + x$ is representable as
distinct from $1$.  Iterative methods compound rounding error,
and Lanczos (Section 5) is the textbook example of compounded
roundoff destroying a beautiful algorithm.

In [ ]:
# Exercise 0.3 (🔴⚪⚪⚪⚪, 3 min)
# Find epsilon_machine empirically for fp32 and fp64.

def find_eps(dtype):
    one = torch.tensor(1.0, dtype=dtype)
    eps = torch.tensor(1.0, dtype=dtype)
    while one + eps / 2 > one:
        eps = eps / 2
    return eps.item()

eps32 = None  # YOUR CODE HERE
eps64 = None  # YOUR CODE HERE
assert eps32 is not None and eps64 is not None
print(f'eps(fp32) ≈ {eps32:.3e};  eps(fp64) ≈ {eps64:.3e}')

### 0.4 Rayleigh quotient & symmetric eigenvalues

For symmetric $A$ and unit $v$:

$$
R(v) = v^\top A v \in [\lambda_{\min}(A), \, \lambda_{\max}(A)].
$$

- $R(v_{\max}) = \lambda_{\max}$ when $v_{\max}$ is the top eigenvector.
- Power iteration is *ascent on $R$*: each step pushes $v$ further in the
  direction of largest $R$.

**Eigenvalues vs singular values.**  For symmetric $A$,
$\sigma_i(A) = |\lambda_i(A)|$ — singular values are absolute values of
eigenvalues.  For the Hessian, this matters: the **largest** eigenvalue and
the **largest-magnitude** eigenvalue can differ if the Hessian is indefinite
(common during training).  Power iteration converges to the largest-magnitude
one.

In [ ]:
# Exercise 0.4 (🔴⚪⚪⚪⚪, 2 min)
# Verify: R(v) is bounded by [lambda_min, lambda_max] for random unit v.

torch.manual_seed(0)
A = torch.randn(15, 15); A = A + A.T
eigs = torch.linalg.eigvalsh(A)
lam_min, lam_max = eigs.min().item(), eigs.max().item()

for _ in range(20):
    v = torch.randn(15); v = v / v.norm()
    R = (v @ A @ v).item()
    assert lam_min - 1e-6 <= R <= lam_max + 1e-6, f'R={R} outside [{lam_min},{lam_max}]'
print(f'all 20 Rayleigh quotients in [{lam_min:.3f}, {lam_max:.3f}] ✓')

## 1. The matvec is the unit of cost

The **Hessian-vector product** (HVP) `H @ v` is the only way we'll touch the
Hessian of a neural network — never the matrix itself.  Three equivalent
implementations, all running in $O(\text{forward pass})$ time.

**Setup:**
- model: `toy_mlp(seed=1)`, ~250 params
- loss: cross-entropy on a small batch

We have three ways to compute $H v$:
1. **Finite differences on the gradient:** $\frac{\nabla \mathcal{L}(\theta + \epsilon v) - \nabla \mathcal{L}(\theta - \epsilon v)}{2\epsilon}$
2. **Double backward (the Pearlmutter trick):** form the scalar $g^\top v$ and differentiate again.
3. **JVP of grad:** push $v$ forward through `grad`, using `torch.func.jvp`.

In [ ]:
# Setup
torch.manual_seed(0)
model = toy_mlp(seed=1)
X = torch.randn(8, 20)
y = torch.randint(0, 4, (8,))
P = count_params(model)
v = torch.randn(P)
print(f'model: {P} params')

### Exercise 1.1: HVP via double backward (🔴🔴⚪⚪⚪, 10 min)

Implement `hvp_double_backward(model, X, y, v)`.  Hint: build a single flat
parameter tensor with `requires_grad=True`, recompute the loss with
`torch.func.functional_call`, take the first gradient with
`create_graph=True`, then take a second gradient of $g \cdot v$.

In [ ]:
import torch.nn.functional as F
from torch.func import functional_call

def flat_params(model):
    return torch.cat([p.detach().reshape(-1) for p in model.parameters()])

def unflatten_into_dict(flat, ref_named):
    out, i = {}, 0
    for n, p in ref_named.items():
        out[n] = flat[i:i+p.numel()].view_as(p)
        i += p.numel()
    return out

def hvp_double_backward(model, X, y, v):
    # YOUR CODE HERE
    raise NotImplementedError

tests.test_hvp(hvp_double_backward)

### Exercise 1.2: HVP via JVP of grad (🔴🔴🔴⚪⚪, 12 min)

Same answer, different machinery.  Use `torch.func.grad` to build a function
that returns the flat gradient, then `torch.func.jvp` to push `v` forward
through it.

The result of `jvp(f, (x,), (v,))` is `(f(x), df_x(v))`.  Use the second.

In [ ]:
from torch.func import grad, jvp

def hvp_jvp_of_grad(model, X, y, v):
    # YOUR CODE HERE
    raise NotImplementedError

tests.test_hvp(hvp_jvp_of_grad)

### Exercise 1.3: HVP via finite differences (🔴⚪⚪⚪⚪, 5 min)

Central differences on `grad`.  Slow, noisy, but the reference-of-references
that we'll use to sanity-check the others.  Use $\epsilon = 10^{-3}$.

In [ ]:
def hvp_finite_difference(model, X, y, v, eps=1e-3):
    # YOUR CODE HERE
    raise NotImplementedError

# Verify all three agree.
v_test = torch.randn(P)
h_dbl = hvp_double_backward(model, X, y, v_test)
h_jvp = hvp_jvp_of_grad(model, X, y, v_test)
h_fd  = hvp_finite_difference(model, X, y, v_test)

print(f'||double - jvp||_inf = {(h_dbl - h_jvp).abs().max():.2e}')
print(f'||double - fd||_inf  = {(h_dbl - h_fd).abs().max():.2e}  (FD is noisier)')
assert torch.allclose(h_dbl, h_jvp, atol=1e-5)
assert torch.allclose(h_dbl, h_fd,  atol=5e-3)
print('all three agree ✓')

### Exercise 1.4: Timing (🔴⚪⚪⚪⚪, 5 min)

Time all three HVPs.  Expected ordering: `jvp_of_grad` ≈ `double_backward`
(both single forward+backward); `finite_difference` ≈ 2× as costly
(two gradient computations).

**Punchline:** an HVP is one forward + one backward, independent of $P$.
That's why we never materialize the Hessian.

In [ ]:
import time

def time_hvp(fn, n_calls=20):
    fn(model, X, y, v)  # warmup
    t0 = time.perf_counter()
    for _ in range(n_calls):
        fn(model, X, y, v)
    return (time.perf_counter() - t0) / n_calls

t_dbl = time_hvp(hvp_double_backward)
t_jvp = time_hvp(hvp_jvp_of_grad)
t_fd  = time_hvp(hvp_finite_difference)
print(f'double backward: {t_dbl*1e3:.2f} ms')
print(f'jvp of grad:     {t_jvp*1e3:.2f} ms')
print(f'finite diff:     {t_fd*1e3:.2f} ms (≈ 2× the others)')